# Validation suite (paper Sec. 4.5, Table 2, Figure 3) + exposure (App. C, Fig. 4)

Eight models: the six-model core plus OLMo 7B and Pythia 2.8B (GPT-NeoX
parallel-block path, certified in `frozen_neox.py`). Resume-friendly; one
model resident at a time; progress bars throughout.

In [ ]:
MODELS = [
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B",
    "meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Llama-3.1-8B",
    "Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-7B",
    "allenai/OLMo-7B-hf",
    "EleutherAI/pythia-2.8b",
]
ONLY   = ""
DEVICE = "cuda"

import os, gc, json, sys, subprocess, torch, pandas as pd
from tqdm.auto import tqdm
torch.set_grad_enabled(False)
models = [m for m in MODELS if ONLY.lower() in m.lower()]
tag = lambda m: m.split("/")[-1].replace(".", "").lower()
print([tag(m) for m in models])

def free():
    import __main__
    for v in ("model", "tok", "W", "api", "C"):
        if hasattr(__main__, v):
            delattr(__main__, v)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 1. Cohorts (60 failures + 60 successes, six PopQA relations)
Prefer copying existing cohorts; missing ones are built here (records now
carry `relation` and `s_pop` for the decoy sets and the exposure analysis).

In [ ]:
from build_cohort import process, load_popqa
from validation_suite import load_arch

def ensure_cohort(m):
    path = f"cohorts/{tag(m)}_parametric.jsonl"
    if os.path.exists(path):
        return path
    os.makedirs("cohorts", exist_ok=True)
    free()
    model, tok, (W, api) = load_arch(m, DEVICE)
    src = load_popqa(170)
    out, nf = [], 0
    bar = tqdm(src, desc=f"{tag(m)} build cohort")
    for r in bar:
        try:
            rec = process(model, tok, DEVICE, r["prompt"], r["subject"],
                          r["answer"], r.get("aliases", []))
        except (ValueError, IndexError):
            continue
        rec["relation"], rec["s_pop"] = r.get("relation"), r.get("s_pop")
        out.append(rec)
        nf += not rec["correct"]
        bar.set_postfix(failures=nf)
        if nf >= 60 and len(out) - nf >= 60:
            break
    clean = [o for o in out if not o["copy_contaminated"]]
    fails = [o for o in clean if not o["correct"]][:60]
    succs = [o for o in clean if o["correct"]][:60]
    with open(path, "w") as f:
        for o in fails + succs:
            f.write(json.dumps(o) + "\n")
    free()
    return path

cohorts = {m: ensure_cohort(m) for m in models}

## 2. Labels + calibrated readout interventions (per model)

In [ ]:
from run_validation_suite import run_model
for m in models:
    out = f"results/{tag(m)}"
    if os.path.exists(f"{out}/validation_suite.csv"):
        print(f"{tag(m)}: exists, skipping")
        continue
    free()
    run_model(m, cohorts[m], out, DEVICE)
    free()

## 3. Aggregate statistics (Table 2, Figure 3 numbers)

In [ ]:
from run_validation_suite import aggregate
aggregate()

In [ ]:
# Figure 3 style plot
import glob, matplotlib.pyplot as plt
from stats_paper import wilson
frames = [pd.read_csv(p).assign(model=p.split(os.sep)[-2])
          for p in glob.glob("results/*/validation_suite.csv")]
if frames:
    df = pd.concat(frames)
    labs = ["source", "transport", "selection"]
    arms = ["answer_up", "combined", "random"]
    fig, ax = plt.subplots(figsize=(7, 3.4))
    width = 0.25
    for j, arm in enumerate(arms):
        xs, ys, errs = [], [], []
        for i, lab in enumerate(labs):
            sub = df[df.label == lab]
            k, n = int(sub[arm].sum()), max(len(sub), 1)
            lo, hi = wilson(k, n)
            xs.append(i + (j - 1) * width)
            ys.append(k / n)
            errs.append([k / n - lo, hi - k / n])
        ax.bar(xs, ys, width, label=arm,
               yerr=list(zip(*errs)), capsize=3,
               color=["#4477aa", "#ee6677", "#66aa66"][j])
    ax.set_xticks(range(3), labs)
    ax.set_ylabel("first-token recovery")
    ax.set_title("calibrated readout interventions by label")
    ax.legend()
    plt.tight_layout()
    plt.savefig("results/F3_readout_interventions.png", dpi=150)
    plt.show()

## 4. Exposure by label (Appendix C, Figure 4)
Infini-gram co-occurrence for the public-corpus models (Pythia -> Pile,
OLMo -> Dolma); PopQA popularity elsewhere. API calls cached; offline runs
fall back to popularity automatically.

In [ ]:
PY = sys.executable
IDX = {"pythia-28b": "v4_piletrain_llama", "olmo-7b-hf": "v4_dolma-v1_7_llama"}
for m in models:
    t = tag(m)
    suite = f"results/{t}/validation_suite.csv"
    if not os.path.exists(suite) or os.path.exists(f"results/{t}/exposure.csv"):
        continue
    cmd = [PY, "run_exposure.py", "--cohort", cohorts[m],
           "--suite", suite, "--out", f"results/{t}"]
    if t in IDX:
        cmd += ["--index", IDX[t]]
    subprocess.run(cmd)

In [ ]:
# Figure 4 style plot + pooled dump
ps = glob.glob("results/*/exposure.csv")
if ps:
    ex = pd.concat([pd.read_csv(p).assign(model=p.split(os.sep)[-2])
                    for p in ps])
    fig, ax = plt.subplots(figsize=(6, 3.2))
    data = [ex[ex.label == l].log_exp.dropna() for l in
            ("source", "transport", "selection")]
    ax.boxplot([d for d in data if len(d)],
               labels=[l for l, d in zip(("source", "transport", "selection"),
                                         data) if len(d)])
    ax.set_ylabel("log10 exposure")
    ax.set_title("training-data exposure by diagnostic category")
    plt.tight_layout()
    plt.savefig("results/F4_exposure.png", dpi=150)
    plt.show()
    display(ex.groupby("label").log_exp.describe().round(2))